In [1]:
from vivarium import Artifact, InteractiveContext
import pandas as pd, numpy as np, os

In [2]:
! pip list | grep vivarium

# [software verion + hash . date]

vivarium                                 4.0.3
vivarium_build_utils                     3.2.2
vivarium_cluster_tools                   3.1.6
vivarium-compat                          0.6.1
vivarium-dependencies                    1.1.0
vivarium_gates_mncnh                     34.1.dev8+g5eaecaf6f.d20260616 /mnt/share/homes/tylerdy/vivarium_gates_mncnh
vivarium-gbd-mapping                     6.0.1
vivarium_public_health                   5.0.2
vivarium-risk-distributions              3.1.1
vivarium_testing_utils                   0.5.5


In [3]:
! pip freeze | grep vivarium

vivarium==4.0.3
vivarium-compat==0.6.1
vivarium-dependencies==1.1.0
vivarium-gbd-mapping==6.0.1
vivarium-risk-distributions==3.1.1
vivarium_build_utils==3.2.2
vivarium_cluster_tools==3.1.6
-e git+https://github.com/ihmeuw/vivarium_gates_mncnh.git@316c7cf9cfbc1c5bec0178c2a5ca9360eccd42fa#egg=vivarium_gates_mncnh
vivarium_public_health==5.0.2
vivarium_testing_utils==0.5.5


In [4]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

from vivarium import InteractiveContext, Artifact

In [ ]:
import vivarium_gates_mncnh
from vivarium.framework.configuration import build_model_specification
from pathlib import Path

from vivarium_gates_mncnh.constants.data_values import (
    SIMULATION_EVENT_NAMES,
    COLUMNS,
    PREGNANCY_OUTCOMES,
    PIPELINES,
)

path = Path(vivarium_gates_mncnh.__file__).parent / 'model_specifications/model_spec.yaml'
custom_model_specification = build_model_specification(path)
custom_model_specification.configuration.input_data.input_draw_number = 60

custom_model_specification.configuration.population.population_size = 20_000 * 3

included_components = custom_model_specification.components.vivarium_gates_mncnh.components
included_components

['Hemoglobin()',
 'AnemiaInterventionPropensity()',
 'AgelessPopulation("population.scaling_factor")',
 'Pregnancy()',
 'ResultsStratifier()',
 'ANCAttendance()',
 'Ultrasound()',
 'MaternalDisorder("maternal_obstructed_labor_and_uterine_rupture")',
 'MaternalDisorder("maternal_sepsis_and_other_maternal_infections")',
 'AntepartumHemorrhage()',
 'PostpartumHemorrhage()',
 'ResidualMaternalDisorders()',
 'AbortionMiscarriageEctopicPregnancy()',
 'MaternalDisordersBurden()',
 'LBWSGRisk()',
 "LBWSGRiskEffect('cause.all_causes.all_cause_mortality_risk')",
 "LBWSGRiskEffect('cause.neonatal_sepsis_and_other_neonatal_infections.cause_specific_mortality_risk')",
 "LBWSGRiskEffect('cause.neonatal_preterm_birth_with_rds.cause_specific_mortality_risk')",
 "LBWSGRiskEffect('cause.neonatal_preterm_birth_without_rds.cause_specific_mortality_risk')",
 "LBWSGRiskEffect('cause.neonatal_encephalopathy_due_to_birth_asphyxia_and_trauma.cause_specific_mortality_risk')",
 "PretermBirth('neonatal_preterm_bi

In [6]:
artifact_path = custom_model_specification.configuration.input_data.artifact_path
art = Artifact(artifact_path)

artifact_path

'/mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/aph/ethiopia.hdf'

In [7]:
draw_num = custom_model_specification.configuration.input_data.input_draw_number
draw = 'draw_' + str(draw_num)
draw

'draw_60'

In [23]:
aph_on_hemoglobin_shift_early = art.load("risk_effect.hemorrhage_hemoglobin_shift.aph_shift_0_6w")[draw][0]
pph_on_hemoglobin_shift_early = art.load("risk_effect.hemorrhage_hemoglobin_shift.pph_shift_0_6w")[draw][0]
aph_on_hemoglobin_shift_late = art.load("risk_effect.hemorrhage_hemoglobin_shift.aph_shift_6w_9m")[draw][0]
pph_on_hemoglobin_shift_late = art.load("risk_effect.hemorrhage_hemoglobin_shift.pph_shift_6w_9m")[draw][0]
art_shifts = {
    "aph_on_hemoglobin_shift_early": aph_on_hemoglobin_shift_early, 
    "pph_on_hemoglobin_shift_early": pph_on_hemoglobin_shift_early, 
    "aph_on_hemoglobin_shift_late": aph_on_hemoglobin_shift_late, 
    "pph_on_hemoglobin_shift_late": pph_on_hemoglobin_shift_late,
}
art_shifts

{'aph_on_hemoglobin_shift_early': -9.319741765293287,
 'pph_on_hemoglobin_shift_early': -16.202410730463864,
 'aph_on_hemoglobin_shift_late': -5.875628017090329,
 'pph_on_hemoglobin_shift_late': -0.6569587720997878}

In [54]:
def check_hemoglobin_shift(sim: InteractiveContext):
    pop = sim.get_population([
        COLUMNS.ANTEPARTUM_HEMORRHAGE, 
        COLUMNS.POSTPARTUM_HEMORRHAGE, 
        COLUMNS.PREGNANCY_OUTCOME,
        COLUMNS.ANEMIA_STATUS_DURING_PREGNANCY,
        ])
    # Check full term with anemia status
    full_term = pop.loc[
        ((pop[COLUMNS.PREGNANCY_OUTCOME] == PREGNANCY_OUTCOMES.FULL_TERM_OUTCOME) |
        (pop[COLUMNS.PREGNANCY_OUTCOME] == PREGNANCY_OUTCOMES.LIVE_BIRTH_OUTCOME) |
        (pop[COLUMNS.PREGNANCY_OUTCOME] == PREGNANCY_OUTCOMES.STILLBIRTH_OUTCOME)) &
        pop[COLUMNS.ANEMIA_STATUS_DURING_PREGNANCY]
    ]
    has_aph = full_term[COLUMNS.ANTEPARTUM_HEMORRHAGE]
    has_pph = full_term[COLUMNS.POSTPARTUM_HEMORRHAGE]
    has_no_hem = ~has_aph & ~has_pph
    if len(full_term) > 0 and has_aph.sum() > 0 and has_pph.sum() > 0:
        # Get hemoglobin exposure values from the pipeline
        hgb = sim.get_population(PIPELINES.HEMOGLOBIN_EXPOSURE)
        anemia_statuses = full_term[COLUMNS.ANEMIA_STATUS_DURING_PREGNANCY].unique()
        hgb_means = {}
        hgb_means["next_step"] = sim._clock.step_name
        hgb_means["aph"] = hgb.loc[full_term.index[has_aph]].mean()
        hgb_means["pph"] = hgb.loc[full_term.index[has_pph]].mean()
        for anemia_status in anemia_statuses:
            has_status = full_term[COLUMNS.ANEMIA_STATUS_DURING_PREGNANCY] == anemia_status
            hgb_means[anemia_status] = hgb.loc[full_term.index[has_status]].mean()
            hgb_means[f"{anemia_status}_aph"] = hgb.loc[full_term.index[has_status & has_aph]].mean()
            hgb_means[f"{anemia_status}_pph"] = hgb.loc[full_term.index[has_status & has_pph]].mean()
            hgb_means[f"{anemia_status}_no_hem"] = hgb.loc[full_term.index[has_status & has_no_hem]].mean()
        return hgb_means

    # TODO check for age groups as well?

In [10]:
def run_all_steps(sim: InteractiveContext):
    step_shifts = []
    while sim.get_number_of_steps_remaining() > 0:
        step_shift = check_hemoglobin_shift(sim)
        if step_shift:
            step_shifts.append(step_shift)
        sim.step()
    return pd.DataFrame(step_shifts)

In [55]:
sim = InteractiveContext(custom_model_specification)
step_shifts = run_all_steps(sim)

2026-07-15 09:55:56.499 | INFO     | simulation_9-artifact_manager:80 - Running simulation from artifact located at /mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/artifacts/aph/ethiopia.hdf.
2026-07-15 09:55:56.501 | INFO     | simulation_9-artifact_manager:81 - Artifact base filter terms are ['draw == 60'].
2026-07-15 09:55:56.506 | INFO     | simulation_9-artifact_manager:82 - Artifact additional filter terms are None.
2026-07-15 09:56:35.832 | WARNING  | simulation_9-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-07-15 09:56:35.833 | WARNING  | simulation_9-results_manager:409 - Specified excluded stratifications are already not included by default: ['stillbirth', 'partial_term']
2026-07-15 09:56:36.005 | WARNING  | simulation_9-lookup_table_manager:85 - Component 'risk_factor.hemoglobin' configured, but didn't build lookup table 'exposure' during setup.
2026-07-15 09:56:36.006 | WARNING

In [56]:
step_shifts

,next_step,aph,pph,not_anemic,not_anemic_aph,not_anemic_pph,not_anemic_no_hem,moderate,moderate_aph,moderate_pph,moderate_no_hem,mild,mild_aph,mild_pph,mild_no_hem,severe,severe_aph,severe_pph,severe_no_hem
0,maternal_sepsis_and_other_maternal_infections,109.933273,111.002264,133.245203,133.175586,133.380925,133.232638,92.214675,89.996292,89.549052,92.902177,108.057420,107.464534,107.220142,108.165799,57.894915,55.729733,54.685022,59.710322
1,residual_maternal_disorders,109.933273,111.002264,133.245203,133.175586,133.380925,133.232638,92.214675,89.996292,89.549052,92.902177,108.057420,107.464534,107.220142,108.165799,57.894915,55.729733,54.685022,59.710322
2,abortion_miscarriage_ectopic_pregnancy,109.933273,111.002264,133.245203,133.175586,133.380925,133.232638,92.214675,89.996292,89.549052,92.902177,108.057420,107.464534,107.220142,108.165799,57.894915,55.729733,54.685022,59.710322
3,mortality,109.933273,111.002264,133.245203,133.175586,133.380925,133.232638,92.214675,89.996292,89.549052,92.902177,108.057420,107.464534,107.220142,108.165799,57.894915,55.729733,54.685022,59.710322
4,early_neonatal_mortality,98.627917,94.308637,131.738060,122.712473,116.872582,133.232638,88.837979,77.746327,72.425504,92.902177,106.132866,96.956615,90.708012,108.165799,51.682077,41.984128,38.011638,59.710322
5,late_neonatal_mortality,109.933273,111.002264,133.245203,133.175586,133.380925,133.232638,92.214675,89.996292,89.549052,92.902177,108.057420,107.464534,107.220142,108.165799,57.894915,55.729733,54.685022,59.710322
6,postpartum_hemoglobin_nine_month,118.344816,124.720745,147.540470,141.600981,147.298498,147.765832,106.103397,98.415928,103.024465,107.372839,122.380718,115.994460,121.178412,122.791431,71.134319,63.863891,67.644989,73.988617
7,postpartum_depression,109.933273,111.002264,133.245203,133.175586,133.380925,133.232638,92.214675,89.996292,89.549052,92.902177,108.057420,107.464534,107.220142,108.165799,57.894915,55.729733,54.685022,59.710322


In [57]:
shift_diffs = step_shifts[["next_step"]].copy()
shift_diffs["not_anemic_aph"] = step_shifts["not_anemic_aph"] - step_shifts["not_anemic_no_hem"]
shift_diffs["not_anemic_pph"] = step_shifts["not_anemic_pph"] - step_shifts["not_anemic_no_hem"]
shift_diffs["mild_aph"] = step_shifts["mild_aph"] - step_shifts["mild_no_hem"]
shift_diffs["mild_pph"] = step_shifts["mild_pph"] - step_shifts["mild_no_hem"]
shift_diffs["moderate_aph"] = step_shifts["moderate_aph"] - step_shifts["moderate_no_hem"]
shift_diffs["moderate_pph"] = step_shifts["moderate_pph"] - step_shifts["moderate_no_hem"]
shift_diffs["severe_aph"] = step_shifts["severe_aph"] - step_shifts["severe_no_hem"]
shift_diffs["severe_pph"] = step_shifts["severe_pph"] - step_shifts["severe_no_hem"]
shift_diffs

,next_step,not_anemic_aph,not_anemic_pph,mild_aph,mild_pph,moderate_aph,moderate_pph,severe_aph,severe_pph
0,maternal_sepsis_and_other_maternal_infections,-0.057051,0.148287,-0.701265,-0.945656,-2.905884,-3.353125,-3.980589,-5.025300
1,residual_maternal_disorders,-0.057051,0.148287,-0.701265,-0.945656,-2.905884,-3.353125,-3.980589,-5.025300
2,abortion_miscarriage_ectopic_pregnancy,-0.057051,0.148287,-0.701265,-0.945656,-2.905884,-3.353125,-3.980589,-5.025300
3,mortality,-0.057051,0.148287,-0.701265,-0.945656,-2.905884,-3.353125,-3.980589,-5.025300
4,early_neonatal_mortality,-10.520164,-16.360056,-11.209184,-17.457786,-15.155849,-20.476673,-17.726194,-21.698684
5,late_neonatal_mortality,-0.057051,0.148287,-0.701265,-0.945656,-2.905884,-3.353125,-3.980589,-5.025300
6,postpartum_hemoglobin_nine_month,-6.164852,-0.467334,-6.796971,-1.613020,-8.956910,-4.348373,-10.124726,-6.343628
7,postpartum_depression,-0.057051,0.148287,-0.701265,-0.945656,-2.905884,-3.353125,-3.980589,-5.025300
